# Rigid molecular assembly with ASE

## 0. How to use this notebook

This guided workspace has five jobs: **(1)** load a monomer, **(2)** define its local molecular frame, **(3)** construct vertical and horizontal assemblies, **(4)** validate the generated geometry, and **(5)** inspect and export the result. Run one small cell at a time and look at the figure immediately below it before changing the next set of parameters.

**USER-EDITABLE PARAMETERS**

- input XYZ path
- `plane_indices`, `x_axis_indices`, and `normal_reference`
- vertical separation, `slip_x`, `slip_y`, and twist angle
- horizontal local translation and rotation

**CORE IMPLEMENTATION**

You normally do not need to edit the frame mathematics, rigid transforms, component metadata, or validation logic. Those live in the project modules imported below. All placements are rigid-body geometry operations in Å and degrees; the notebook does not optimise structures or infer chemical interaction sites.

## A. Setup

The project-directory search works when the notebook is launched from the notebook folder, the project folder, or the repository root. `ENABLE_GUI` stays `False` so every required cell can run in a headless environment.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from ase.data import covalent_radii
from ase.data.colors import jmol_colors
from ase.io import read, write
from ase.visualize import view
from ase.visualize.plot import plot_atoms


def find_project_dir(start: Path) -> Path:
    candidates = (start, start.parent, start / "ase-molecular-assembly")
    for candidate in candidates:
        if (candidate / "frame.py").is_file() and (candidate / "assembly.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate ase-molecular-assembly. Launch this notebook from its "
        "notebook folder, the project folder, or the repository root."
    )


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from assembly import (
    Component,
    LateralPlacement,
    VerticalInterface,
    add_lateral_components,
    build_horizontal_assembly,
    build_vertical_stack,
)
from frame import define_frame
from validation import validate_assembly

OUTPUT_DIR = PROJECT_DIR / "notebook" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENABLE_GUI = False

print("Project modules located.")
print("GUI enabled:", ENABLE_GUI)

The next two helpers are presentation-only. `show_structure` uses ASE's Matplotlib renderer for headless inline figures. `show_frame` adds a lightweight 3D axis overlay. Assembly plots may colour atoms by `component_id` purely to distinguish rigid bodies; these colours do not encode elements or chemical properties.

In [ ]:
def show_structure(
    atoms,
    title,
    *,
    rotation="0x,0y,0z",
    figsize=(6.0, 4.5),
    ax=None,
    color_by_component=False,
):
    """Render an ASE Atoms object inline without requiring a GUI."""
    created_figure = ax is None
    if created_figure:
        _, ax = plt.subplots(figsize=figsize)

    colors = None
    if color_by_component and "component_id" in atoms.arrays:
        palette = plt.get_cmap("tab10")
        colors = [palette(int(component_id) % 10) for component_id in atoms.arrays["component_id"]]

    plot_atoms(
        atoms,
        ax=ax,
        rotation=rotation,
        show_unit_cell=0,
        radii=0.70,
        colors=colors,
    )
    ax.set_title(title)
    if created_figure:
        plt.tight_layout()
        plt.show()
    return ax


def show_frame(atoms, frame, *, axis_length=2.5):
    """Show a molecular frame beside the atoms in global Cartesian space."""
    positions = atoms.get_positions()
    fig = plt.figure(figsize=(7.0, 5.5))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(
        positions[:, 0],
        positions[:, 1],
        positions[:, 2],
        s=180.0 * covalent_radii[atoms.numbers] ** 2,
        c=jmol_colors[atoms.numbers],
        edgecolors="black",
        linewidths=0.4,
        depthshade=True,
    )

    axes = (
        ("+x", frame.x_axis, "tab:red"),
        ("+y", frame.y_axis, "tab:green"),
        ("+normal", frame.normal, "tab:blue"),
    )
    endpoints = []
    for label, direction, color in axes:
        vector = axis_length * direction
        endpoint = frame.origin + vector
        endpoints.append(endpoint)
        ax.quiver(*frame.origin, *vector, color=color, linewidth=2.2, arrow_length_ratio=0.14)
        ax.text(*endpoint, label, color=color, fontsize=10, weight="bold")
    ax.scatter(*frame.origin, color="black", s=35, label="frame origin")

    extent_points = np.vstack([positions, frame.origin, *endpoints])
    midpoint = 0.5 * (extent_points.min(axis=0) + extent_points.max(axis=0))
    radius = 0.55 * np.ptp(extent_points, axis=0).max()
    ax.set_xlim(midpoint[0] - radius, midpoint[0] + radius)
    ax.set_ylim(midpoint[1] - radius, midpoint[1] + radius)
    ax.set_zlim(midpoint[2] - radius, midpoint[2] + radius)
    ax.set_xlabel("global X / Å")
    ax.set_ylabel("global Y / Å")
    ax.set_zlabel("global Z / Å")
    ax.set_title("p-Toluic acid with its local molecular frame")
    ax.view_init(elev=24, azim=-62)
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

## B. Load and inspect the monomer

This geometry dogfooding example uses p-toluic acid from the repository-relative `dogfooding_p-toluic-acid` folder. Its aromatic ring gives a clear candidate molecular plane, while its methyl and carboxylic-acid substituents introduce directional asymmetry that makes frame orientation and rigid transformations easy to see. No energetic or interaction claim is implied.

In [ ]:
EXAMPLE_XYZ = PROJECT_DIR / "dogfooding_p-toluic-acid" / "p-toluic-acid.xyz"
if not EXAMPLE_XYZ.is_file():
    expected = EXAMPLE_XYZ.relative_to(PROJECT_DIR)
    raise FileNotFoundError(
        f"Expected the tutorial structure at {expected}. "
        "Restore that example file or update EXAMPLE_XYZ to another XYZ structure."
    )

monomer_a = read(EXAMPLE_XYZ)
monomer_b = monomer_a.copy()

print("Input:", EXAMPLE_XYZ.relative_to(PROJECT_DIR))
print("Formula:", monomer_a.get_chemical_formula())
print("Atom count:", len(monomer_a))
show_structure(monomer_a, "Input monomer: p-toluic acid");

### Inspect atom indices before defining a frame

ASE does not know which atoms form a chemically meaningful π-core. Inspect the structure and choose explicitly. In this file, atoms `0–5` are the six aromatic-ring carbons and are the intended `plane_indices`. The ordered pair `x_axis_indices=(i, j)` sets positive local x from atom $i \rightarrow j$ after projection into the fitted plane. `normal_reference` resolves which of the two plane-normal directions is positive.

In [ ]:
print(f"{'index':>5}  {'element':>7}  {'x / Å':>11}  {'y / Å':>11}  {'z / Å':>11}")
for index, atom in enumerate(monomer_a):
    x, y, z = atom.position
    print(f"{index:5d}  {atom.symbol:>7}  {x:11.5f}  {y:11.5f}  {z:11.5f}")

## C. Define the local molecular frame

The library fits a best-fit plane through the selected ring atoms, projects the ordered atom-pair direction into that plane, and derives local y to make a right-handed basis. Edit only the three selection parameters below when adapting the workflow to another monomer.

In [ ]:
plane_indices = [0, 1, 2, 3, 4, 5]
x_axis_indices = (0, 1)
normal_reference = (0.0, 0.0, 1.0)

frame_a = define_frame(
    monomer_a,
    plane_indices,
    x_axis_indices,
    normal_reference=normal_reference,
)
frame_b = define_frame(
    monomer_b,
    plane_indices,
    x_axis_indices,
    normal_reference=normal_reference,
)
component_a = Component(monomer_a, frame_a, label="A", source="p-toluic-acid.xyz")
component_b = Component(monomer_b, frame_b, label="B", source="p-toluic-acid.xyz copy")

print("origin:", frame_a.origin)
print("x axis:", frame_a.x_axis)
print("y axis:", frame_a.y_axis)
print("normal:", frame_a.normal)
print("basis.T @ basis:\n", np.round(frame_a.basis.T @ frame_a.basis, 12))
print(
    "cross(x_axis, y_axis) = normal:",
    np.allclose(np.cross(frame_a.x_axis, frame_a.y_axis), frame_a.normal),
)

`cross(x_axis, y_axis) = normal` means the three unit vectors form a right-handed coordinate system. Later, `slip_x` and `slip_y` follow the red and green in-plane arrows, while vertical separation and twist use the blue +normal direction.

In [ ]:
show_frame(monomer_a, frame_a)

## D. Vertical assembly

Vertical placement is defined in the local molecular frame, not by assuming that global Z is perpendicular to the molecule. Start with the simplest baseline, then add in-plane displacement and twist.

### D1. Cofacial A–A dimer

This baseline places a second rigid copy 3.4 Å along +normal with zero slip and zero twist. It is a geometric reference, not a claim that this orientation is an optimised physical minimum. Look for the two molecular planes directly above one another.

In [ ]:
COFACIAL_SEPARATION = 3.4
cofacial_interface = VerticalInterface(
    normal_separation=COFACIAL_SEPARATION,
    slip_x=0.0,
    slip_y=0.0,
    twist_degrees=0.0,
)
cofacial_dimer = build_vertical_stack(
    [component_a, component_a], [cofacial_interface]
)
print("Atom count:", len(cofacial_dimer))
show_structure(
    cofacial_dimer,
    "Cofacial A–A dimer: 3.4 Å normal separation",
    rotation="-75x,10y,-20z",
    color_by_component=True,
);

### D2. Slip and twist

Normal separation controls displacement perpendicular to the fitted plane. `slip_x` and `slip_y` move the partner within that plane, and `twist_degrees` rotates it around the stacking normal. Compare the baseline and modified dimer side by side; these are the main parameters to explore.

In [ ]:
SLIP_X = 1.0
SLIP_Y = 0.4
TWIST_DEGREES = 22.0
slipped_interface = VerticalInterface(
    normal_separation=3.4,
    slip_x=SLIP_X,
    slip_y=SLIP_Y,
    twist_degrees=TWIST_DEGREES,
)
slipped_dimer = build_vertical_stack(
    [component_a, component_a], [slipped_interface]
)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.6))
show_structure(
    cofacial_dimer,
    "Cofacial baseline",
    rotation="-75x,10y,-20z",
    ax=axes[0],
    color_by_component=True,
)
show_structure(
    slipped_dimer,
    f"Slip ({SLIP_X:.1f}, {SLIP_Y:.1f}) Å; twist {TWIST_DEGREES:.0f}°",
    rotation="-75x,10y,-20z",
    ax=axes[1],
    color_by_component=True,
)
fig.tight_layout()
plt.show()

### D3. Trimer

An ordered $N$-component stack uses exactly $N-1$ interfaces. Interface 0 places component 1 relative to component 0; interface 1 then places component 2 relative to component 1. The values below are therefore relative interface instructions, not absolute coordinates.

In [ ]:
trimer_interfaces = [
    VerticalInterface(3.8, slip_x=0.6, slip_y=0.0, twist_degrees=10.0),
    VerticalInterface(3.8, slip_x=-0.2, slip_y=0.3, twist_degrees=-8.0),
]
aaa_trimer = build_vertical_stack(
    [component_a, component_a, component_a], trimer_interfaces
)
print("Interfaces:", len(trimer_interfaces), "for", 3, "components")
print("Component IDs:", np.unique(aaa_trimer.arrays["component_id"]))
show_structure(
    aaa_trimer,
    "A–A–A trimer from two relative interfaces",
    rotation="-75x,10y,-20z",
    color_by_component=True,
);

### D4. Ordered A–B stack

Here A and B are identical copies of the same p-toluic-acid input. The example demonstrates ordered heterogeneous API semantics—each sequence position may hold its own `Component`—rather than chemically distinct monomers. Replace B with a separately loaded and framed structure when a genuinely heterogeneous model is needed.

In [ ]:
ab_interface = VerticalInterface(
    normal_separation=3.6, slip_x=0.8, slip_y=0.2, twist_degrees=25.0
)
ab_dimer = build_vertical_stack([component_a, component_b], [ab_interface])
labels = [item["label"] for item in ab_dimer.info["assembly"]["components"]]
print("Ordered component labels:", labels)
show_structure(
    ab_dimer,
    "Ordered A–B API example (same molecular geometry)",
    rotation="-75x,10y,-20z",
    color_by_component=True,
);

## E. Horizontal assembly

Horizontal placement also uses the explicit molecular frame. It does not recognise donors, acceptors, hydrogen bonds, or end groups.

### E1. One lateral partner

`translation_local=(x, y, normal)` is expressed in the **host local molecular frame**, not in global Cartesian coordinates. The rotation is an explicit rigid-body rotation about the host normal. The 10 Å local-x displacement below keeps this demonstration visibly separated rather than pretending to infer a preferred intermolecular contact.

In [ ]:
LATERAL_TRANSLATION = (10.0, 0.0, 0.0)
LATERAL_ROTATION = 30.0
lateral_dimer = build_horizontal_assembly(
    component_a,
    [
        LateralPlacement(
            component_b,
            translation_local=LATERAL_TRANSLATION,
            rotation_degrees=LATERAL_ROTATION,
        )
    ],
)
print("Host-frame translation / Å:", LATERAL_TRANSLATION)
print("Rotation about host normal / degrees:", LATERAL_ROTATION)
show_structure(
    lateral_dimer,
    "One host + one explicitly placed lateral partner",
    color_by_component=True,
);

### E2. Multiple lateral partners

The same API places several copies—or separately defined components—around one host. The result is deterministic rigid placement, not an automatically optimised supramolecular geometry.

In [ ]:
lateral_cluster = build_horizontal_assembly(
    component_a,
    [
        LateralPlacement(component_a, (10.0, 0.0, 0.0), 20.0),
        LateralPlacement(component_b, (-10.0, 0.0, 0.0), -20.0),
        LateralPlacement(component_a, (0.0, 10.0, 0.0), 90.0),
    ],
)
print("Component IDs:", np.unique(lateral_cluster.arrays["component_id"]))
show_structure(
    lateral_cluster,
    "One host with three lateral partners",
    figsize=(8.0, 6.0),
    color_by_component=True,
);

## F. Mixed assembly

Mixed motifs are composed from existing primitives: first build a vertical stack, then add lateral components. Compare the input trimer and final motif to see exactly what the second operation adds. Existing component IDs are retained and the guest receives the next ID.

In [ ]:
mixed_assembly = add_lateral_components(
    aaa_trimer,
    frame_a,
    [LateralPlacement(component_b, (10.0, 0.0, 0.0), 60.0)],
)
print("Assembly type:", mixed_assembly.info["assembly"]["assembly_type"])
print("Component IDs:", np.unique(mixed_assembly.arrays["component_id"]))

fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.0))
show_structure(
    aaa_trimer,
    "Step 1: vertical A–A–A stack",
    rotation="-75x,10y,-20z",
    ax=axes[0],
    color_by_component=True,
)
show_structure(
    mixed_assembly,
    "Step 2: add one lateral component",
    rotation="-65x,15y,-20z",
    ax=axes[1],
    color_by_component=True,
)
fig.tight_layout()
plt.show()

## G. Validation

Read the main report fields as follows:

- `passed`: overall verdict from the configured geometric checks
- `maximum_rigid_body_error`: largest change in any intra-component pairwise distance after transformation, in Å
- `minimum_intercomponent_distance`: closest contact between atoms belonging to different components, in Å
- `requested_geometry_errors`: values recovered from the assembled component frames minus the requested vertical parameters (reported as signed differences)

A pass means that the requested construction is internally consistent under these checks. It does **not** show that the structure is chemically stable or energetically favourable.

In [ ]:
trimer_report = validate_assembly(
    aaa_trimer,
    [monomer_a, monomer_a, monomer_a],
    reference_frames=[frame_a, frame_a, frame_a],
    vertical_interfaces=trimer_interfaces,
)
mixed_report = validate_assembly(
    mixed_assembly,
    [monomer_a, monomer_a, monomer_a, monomer_b],
)
assert trimer_report.passed, trimer_report.errors
assert mixed_report.passed, mixed_report.errors
assert mixed_report.minimum_intercomponent_distance > 2.0

print("Trimer passed:", trimer_report.passed)
print("Maximum trimer rigid-body error / Å:", trimer_report.maximum_rigid_body_error)
print("Maximum requested-geometry error:", max(abs(value) for value in trimer_report.requested_geometry_errors.values()))
print("Mixed assembly passed:", mixed_report.passed)
print("Minimum mixed inter-component distance / Å:", mixed_report.minimum_intercomponent_distance)

show_structure(
    mixed_assembly,
    f"Validated mixed assembly (closest contact: {mixed_report.minimum_intercomponent_distance:.2f} Å)",
    rotation="-65x,15y,-20z",
    figsize=(8.0, 5.5),
    color_by_component=True,
);

### Deliberate clash demonstration

For contrast, the next cell places a guest only 0.05 Å away in local x. This intentionally impossible overlap should fail. It is kept separate from the valid tutorial structures so the warning is unambiguous.

In [ ]:
clashing_dimer = build_horizontal_assembly(
    component_a,
    [LateralPlacement(component_a, (0.05, 0.0, 0.0), 0.0)],
)
clash_report = validate_assembly(clashing_dimer, [monomer_a, monomer_a])
assert not clash_report.passed
print("Passed:", clash_report.passed)
print("Status:", clash_report.status)
print("Closest contact / Å:", clash_report.minimum_intercomponent_distance)
print("Validation error:", clash_report.errors[0])
show_structure(
    clashing_dimer,
    "Intentional overlap: validation must fail",
    color_by_component=True,
);

## H. Export and reload

Plain XYZ stores elements and coordinates. Extended XYZ can additionally preserve ASE arrays such as the deterministic per-atom `component_id`. Export the validated mixed structure to the notebook's ignored local output directory, reload the extXYZ, verify the IDs, and inspect the round trip visually.

In [ ]:
xyz_path = OUTPUT_DIR / "mixed_assembly.xyz"
extxyz_path = OUTPUT_DIR / "mixed_assembly.extxyz"
write(xyz_path, mixed_assembly, format="xyz")
write(extxyz_path, mixed_assembly, format="extxyz")
reloaded_assembly = read(extxyz_path)

assert np.array_equal(
    reloaded_assembly.arrays["component_id"],
    mixed_assembly.arrays["component_id"],
)
print("XYZ:", xyz_path.relative_to(PROJECT_DIR))
print("extXYZ:", extxyz_path.relative_to(PROJECT_DIR))
print("Reloaded component IDs:", np.unique(reloaded_assembly.arrays["component_id"]))
show_structure(
    reloaded_assembly,
    "Reloaded extXYZ: component IDs preserved",
    rotation="-65x,15y,-20z",
    figsize=(8.0, 5.5),
    color_by_component=True,
);

## I. Optional interactive ASE viewer

ASE GUI is useful for rotating the final structure interactively, inspecting contacts, and performing a visual sanity check. It is optional and not required for notebook execution. Set `ENABLE_GUI = True` in the setup cell only when a graphical display is available.

In [ ]:
final_structure = reloaded_assembly
if ENABLE_GUI:
    view(final_structure)
else:
    print("GUI skipped. All required inspection figures were rendered inline.")